In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import random

# =========================================================
# Setup
# =========================================================
ROOT = Path("..")
DATA_ROOT = ROOT / "data"

HAM_DIR = DATA_ROOT / "HAM10000"
HAM_CSV = HAM_DIR / "HAM10000.csv"
IMAGE_DIR = HAM_DIR / "images"   # change if needed
MASK_DIR = DATA_ROOT / "HAM10000_segmentations_lesion_tschandl"

print("HAM_CSV  :", HAM_CSV, "| exists:", HAM_CSV.exists())
print("IMAGE_DIR:", IMAGE_DIR, "| exists:", IMAGE_DIR.exists())
print("MASK_DIR :", MASK_DIR, "| exists:", MASK_DIR.exists())

# =========================================================
# Load original HAM csv
# =========================================================
ham_df = pd.read_csv(HAM_CSV)
print("\nLoaded HAM CSV")
print("Shape:", ham_df.shape)
print("Columns:", list(ham_df.columns))

required_cols = ["image_id", "label", "dx", "split", "image"]
missing = [c for c in required_cols if c not in ham_df.columns]
if missing:
    raise ValueError(f"Missing required columns in HAM CSV: {missing}")

# clean split typo if present
ham_df["split"] = ham_df["split"].replace({"trainisic": "train"})

# =========================================================
# Build segmentation index directly from mask folder
# =========================================================
suffix = "_segmentation.png"
mask_rows = []

pngs = sorted(MASK_DIR.glob("*.png"))
print("\nFound segmentation pngs:", len(pngs))

for p in pngs:
    fname = p.name
    if fname.endswith(suffix):
        image_id = fname[:-len(suffix)]
    else:
        image_id = p.stem

    mask_rows.append({
        "image_id": image_id,
        "mask_filename": fname,
    })

mask_df = pd.DataFrame(mask_rows)

print("Indexed masks:", len(mask_df))
print("Unique image_ids in mask folder:", mask_df["image_id"].nunique())

dups = mask_df[mask_df.duplicated(subset=["image_id"], keep=False)]
print("Duplicate image_ids in mask folder:", len(dups))

# =========================================================
# Merge HAM CSV with mask index
# =========================================================
df = ham_df.merge(mask_df, on="image_id", how="left")
df["mask_found"] = df["mask_filename"].notna().astype(int)

# relative paths only
df["image_rel_path"] = "images/" + df["image"].astype(str)
df["mask_rel_path"] = "../HAM10000_segmentations_lesion_tschandl/" + df["mask_filename"].astype(str)

# absolute paths only for runtime checks
df["image_abs_path"] = df["image_rel_path"].apply(lambda x: (HAM_DIR / x).resolve())
df["mask_abs_path"] = df["mask_rel_path"].apply(lambda x: (HAM_DIR / x).resolve())

# =========================================================
# Keep only requested columns and save overlap CSV
# =========================================================
final_cols = [
    "image_id",
    "label",
    "dx",
    "split",
    "image_rel_path",
    "mask_rel_path",
    "mask_found",
]

final_df = df[final_cols].copy()

out_dir = ROOT / "data" / "HAM10000"
out_dir.mkdir(parents=True, exist_ok=True)

overlap_csv_path = out_dir / "ham_segmentation_overlap.csv"
final_df.to_csv(overlap_csv_path, index=False)

print("\nSaved overlap CSV:", overlap_csv_path)
display(final_df.head())

# =========================================================
# 1) Coverage per split
# =========================================================
coverage = (
    final_df.groupby("split", dropna=False)
    .agg(
        total_images=("image_id", "count"),
        masks_found=("mask_found", "sum")
    )
    .reset_index()
)
coverage["coverage_pct"] = (100 * coverage["masks_found"] / coverage["total_images"]).round(2)

# =========================================================
# 2) No accidental missing files
# =========================================================
df["image_exists"] = df["image_abs_path"].apply(lambda p: Path(p).exists())
df["mask_exists"] = df["mask_abs_path"].apply(lambda p: Path(p).exists())

exist_summary = pd.DataFrame({
    "n_rows": [len(df)],
    "image_exists_sum": [int(df["image_exists"].sum())],
    "mask_exists_sum": [int(df["mask_exists"].sum())],
    "image_missing": [int((~df["image_exists"]).sum())],
    "mask_missing": [int((~df["mask_exists"]).sum())],
})

# =========================================================
# 3) Image-mask loading
# =========================================================
random.seed(42)
valid_df = df[(df["image_exists"]) & (df["mask_exists"])].copy()
n_samples = min(5, len(valid_df))
sample_df = valid_df.sample(n=n_samples, random_state=42).copy()

sample_rows = []

for _, row in sample_df.iterrows():
    image_path = Path(row["image_abs_path"])
    mask_path = Path(row["mask_abs_path"])

    try:
        with Image.open(image_path) as img:
            img_size = img.size
            img_mode = img.mode
        image_ok = True
    except Exception:
        img_size = None
        img_mode = None
        image_ok = False

    try:
        with Image.open(mask_path) as msk:
            mask_size = msk.size
            mask_mode = msk.mode
        mask_ok = True
    except Exception:
        mask_size = None
        mask_mode = None
        mask_ok = False

    sample_rows.append({
        "image_id": row["image_id"],
        "image_ok": image_ok,
        "mask_ok": mask_ok,
        "image_size_wh": img_size,
        "mask_size_wh": mask_size,
        "image_mode": img_mode,
        "mask_mode": mask_mode,
    })

sample_check_df = pd.DataFrame(sample_rows)

# =========================================================
# 4) Resolution compatibility
# =========================================================
size_rows = []

for _, row in valid_df.iterrows():
    image_path = Path(row["image_abs_path"])
    mask_path = Path(row["mask_abs_path"])

    try:
        with Image.open(image_path) as img:
            img_w, img_h = img.size
        with Image.open(mask_path) as msk:
            mask_w, mask_h = msk.size

        size_rows.append({
            "image_id": row["image_id"],
            "image_w": img_w,
            "image_h": img_h,
            "mask_w": mask_w,
            "mask_h": mask_h,
            "size_match": int((img_w == mask_w) and (img_h == mask_h))
        })
    except Exception:
        size_rows.append({
            "image_id": row["image_id"],
            "image_w": None,
            "image_h": None,
            "mask_w": None,
            "mask_h": None,
            "size_match": 0
        })

size_df = pd.DataFrame(size_rows)

size_summary = pd.DataFrame({
    "checked_rows": [len(size_df)],
    "exact_size_match": [int(size_df["size_match"].sum())],
    "size_mismatch": [int((size_df["size_match"] == 0).sum())],
})
size_summary["match_pct"] = (100 * size_summary["exact_size_match"] / size_summary["checked_rows"]).round(2)

# =========================================================
# 5) Binary mask sanity
# =========================================================
def get_unique_values(mask_path):
    with Image.open(mask_path) as msk:
        arr = np.array(msk)
    return np.unique(arr)

binary_rows = []

for _, row in valid_df.iterrows():
    mask_path = Path(row["mask_abs_path"])
    try:
        uniq = get_unique_values(mask_path)
        uniq_set = set(uniq.tolist())
        is_binary_like = uniq_set.issubset({0, 1}) or uniq_set.issubset({0, 255})

        binary_rows.append({
            "image_id": row["image_id"],
            "n_unique_values": len(uniq),
            "min_val": int(np.min(uniq)),
            "max_val": int(np.max(uniq)),
            "is_binary_like": int(is_binary_like),
            "unique_values_preview": str(uniq[:10].tolist())
        })
    except Exception as e:
        binary_rows.append({
            "image_id": row["image_id"],
            "n_unique_values": None,
            "min_val": None,
            "max_val": None,
            "is_binary_like": 0,
            "unique_values_preview": f"ERROR: {e}"
        })

binary_df = pd.DataFrame(binary_rows)

binary_summary = pd.DataFrame({
    "checked_masks": [len(binary_df)],
    "binary_like_masks": [int(binary_df["is_binary_like"].sum())],
    "non_binary_like_masks": [int((binary_df["is_binary_like"] == 0).sum())],
})
binary_summary["binary_like_pct"] = (
    100 * binary_summary["binary_like_masks"] / binary_summary["checked_masks"]
).round(2)

# =========================================================
# Final summary
# =========================================================
print("\n===== FINAL SANITY SUMMARY =====")

print("\n1) Coverage per split")
display(coverage)

print("\n2) Missing files")
display(exist_summary)

print("\n3) Random load check")
display(sample_check_df)

print("\n4) Resolution summary")
display(size_summary)

print("\n5) Binary mask summary")
display(binary_summary)

HAM_CSV  : ../data/HAM10000/HAM10000.csv | exists: True
IMAGE_DIR: ../data/HAM10000/images | exists: True
MASK_DIR : ../data/HAM10000_segmentations_lesion_tschandl | exists: True

Loaded HAM CSV
Shape: (10015, 13)
Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'dataset', 'split', 'label', 'image', 'binary_label', 'age_group']

Found segmentation pngs: 10015
Indexed masks: 10015
Unique image_ids in mask folder: 10015
Duplicate image_ids in mask folder: 0

Saved overlap CSV: ../data/HAM10000/ham_segmentation_overlap.csv


,image_id,label,dx,split,image_rel_path,mask_rel_path,mask_found
0,ISIC_0027419,2,bkl,train,images/ISIC_0027419.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
1,ISIC_0026769,2,bkl,train,images/ISIC_0026769.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
2,ISIC_0025661,2,bkl,train,images/ISIC_0025661.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
3,ISIC_0031633,2,bkl,train,images/ISIC_0031633.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
4,ISIC_0027850,2,bkl,train,images/ISIC_0027850.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1



===== FINAL SANITY SUMMARY =====

1) Coverage per split


,split,total_images,masks_found,coverage_pct
0,test,1232,1232,100.0
1,train,8208,8208,100.0
2,val,575,575,100.0



2) Missing files


...

,image_id,image_ok,mask_ok,image_size_wh,mask_size_wh,image_mode,mask_mode
0,ISIC_0027832,True,True,"(600, 450)","(600, 450)",RGB,L
1,ISIC_0025209,True,True,"(600, 450)","(600, 450)",RGB,L
2,ISIC_0029998,True,True,"(600, 450)","(600, 450)",RGB,L
3,ISIC_0033068,True,True,"(600, 450)","(600, 450)",RGB,L
4,ISIC_0025261,True,True,"(600, 450)","(600, 450)",RGB,L



4) Resolution summary


,checked_rows,exact_size_match,size_mismatch,match_pct
0,10015,10015,0,100.0



5) Binary mask summary


,checked_masks,binary_like_masks,non_binary_like_masks,binary_like_pct
0,10015,10015,0,100.0
